In [1]:
# --- Setup ---------------------------------------------------------------
!pip install -q "ultralytics==8.4.174" "sahi==0.12.8" pycocotools "git+https://github.com/ultralytics/CLIP.git"
import ultralytics, sahi, torch
print(ultralytics.__version__, sahi.__version__, torch.__version__, torch.cuda.device_count(), "GPU(s)")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
8.4.174 0.12.8 2.11.0+cu128 2 GPU(s)


In [2]:
# --- Find your paths (run this, then copy what it prints into Config) -------
# Lists every data.yaml, VisDrone annotation folder and .pt file under your attached Kaggle inputs,
# and reads K1's exact training settings out of its checkpoint (Ultralytics stores them in best.pt).
import os, yaml, torch
from pathlib import Path
ROOT = Path("/kaggle/input")

print("== data.yaml files (pick the one for K1's tile dataset) ==")
for p in ROOT.rglob("*.yaml"):
    try: d = yaml.safe_load(open(p))
    except Exception: continue
    if isinstance(d, dict) and "names" in d:
        names = list(d["names"].values()) if isinstance(d["names"], dict) else d["names"]
        print(f"  {p}\n      names={names[:3]}...  train={d.get('train')}")

print("\n== VisDrone image/annotation folders ==")
for p in sorted(ROOT.rglob("annotations")):
    if p.is_dir() and (p.parent / "images").is_dir():
        print(f"  {p.parent / 'images'}   ({len(os.listdir(p.parent / 'images'))} imgs)\n  {p}")

print("\n== .pt checkpoints (K1 is the YOLO26m one) ==")
for p in ROOT.rglob("*.pt"):
    try: ck = torch.load(p, map_location="cpu", weights_only=False)
    except Exception as e: print(f"  {p}  (could not read: {e})"); continue
    a = ck.get("train_args", {}) if isinstance(ck, dict) else {}
    print(f"  {p}\n      model={a.get('model')}  data={a.get('data')}  imgsz={a.get('imgsz')}  epochs={a.get('epochs')}  batch={a.get('batch')}")

def k1_hp_from_ckpt(path):
    # Pull K1's batch + augmentation settings so E2 trains under the same recipe.
    a = torch.load(path, map_location="cpu", weights_only=False)["train_args"]
    keys = ["batch", "workers", "mosaic", "close_mosaic", "mixup", "copy_paste", "degrees", "translate", "scale",
            "shear", "perspective", "flipud", "fliplr", "hsv_h", "hsv_s", "hsv_v", "erasing", "multi_scale", "cos_lr", "lrf"]
    return {k: a[k] for k in keys if k in a}
# After you know K1's path:  print(k1_hp_from_ckpt("/kaggle/input/.../best.pt"))

== data.yaml files (pick the one for K1's tile dataset) ==

== VisDrone image/annotation folders ==

== .pt checkpoints (K1 is the YOLO26m one) ==
  /kaggle/input/notebooks/ciiphersiren/visdrone-training/yolo26m.pt
      model=yolo26m-objv1-150.pt  data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  imgsz=640  epochs=80  batch=128
  /kaggle/input/notebooks/ciiphersiren/visdrone-training/yolo26n.pt
      model=yolo26n-objv1-150.pt  data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  imgsz=640  epochs=245  batch=128
  /kaggle/input/notebooks/ciiphersiren/visdrone-training/weights/yolo26n.pt
      model=yolo26n-objv1-150.pt  data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  imgsz=640  epochs=245  batch=128
  /kaggle/input/notebooks/ciiphersiren/visdrone-training/runs/K1_y26m_tiles/weights/last.pt
      model=yolo26m.pt  data=/tmp/VisDrone/d2_tiles.yaml  imgsz=1024  epochs=12  batch=16
  /kaggle/input/notebooks/ciiphersiren/visdrone-trainin

In [3]:
from pathlib import Path
from ultralytics.utils import ASSETS_URL
from ultralytics.utils.downloads import download
VD = Path("/tmp/VisDrone_raw")
download([f"{ASSETS_URL}/VisDrone2019-DET-{s}.zip" for s in ("val", "test-dev")], dir=VD, threads=2)
print(*sorted(VD.glob("*/*")), sep="\n")

Unzipping /tmp/VisDrone_raw/VisDrone2019-DET-val.zip to /tmp/VisDrone_raw/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 1.7Kfiles/s 0.6s
Unzipping /tmp/VisDrone_raw/VisDrone2019-DET-test-dev.zip to /tmp/VisDrone_raw/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 1.7Kfiles/s 1.9s
/tmp/VisDrone_raw/VisDrone2019-DET-test-dev/annotations
/tmp/VisDrone_raw/VisDrone2019-DET-test-dev/images
/tmp/VisDrone_raw/VisDrone2019-DET-val/annotations
/tmp/VisDrone_raw/VisDrone2019-DET-val/images


In [4]:
# --- Build K1's training data (D2 = ignore-masked full images + 640/960 tiles) ---
# Copied from the K1 training notebook (visdrone-training, Cells 2-3) with only the test-challenge
# download and the stats printout removed. Tiling is seeded per image, so this reproduces K1's data exactly:
# expect "D1: 6471 masked images | D2 tiles: 63745". Takes ~5 min, ~15 GB in /tmp.
import os, json, zipfile, urllib.request, cv2, random, shutil, numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

DATA = Path("/tmp/VisDrone"); DATA.mkdir(parents=True, exist_ok=True)
BASE = "https://github.com/ultralytics/assets/releases/download/v0.0.0/"
SPLITS = {"train": "VisDrone2019-DET-train", "val": "VisDrone2019-DET-val", "test-dev": "VisDrone2019-DET-test-dev"}
NAMES = ["pedestrian", "people", "bicycle", "car", "van",
         "truck", "tricycle", "awning-tricycle", "bus", "motor"]

def download_split(name):
    z = DATA / f"{name}.zip"
    if (DATA / name).exists():
        return True
    print("Downloading", name)
    urllib.request.urlretrieve(BASE + z.name, z)
    with zipfile.ZipFile(z) as f:
        f.extractall(DATA)
    z.unlink()
    return True

def convert(split_dir):
    img_dir, ann_dir = split_dir / "images", split_dir / "annotations"
    lbl_dir = split_dir / "labels"
    lbl_dir.mkdir(exist_ok=True)
    ignores = {}
    for ann in tqdm(sorted(ann_dir.glob("*.txt")), desc=split_dir.name):
        W, H = Image.open(img_dir / f"{ann.stem}.jpg").size
        out, ign = [], []
        for row in ann.read_text().strip().splitlines():
            v = row.strip().rstrip(",").split(",")
            if len(v) < 6:
                continue
            x, y, w, h, score, cat = map(int, v[:6])
            if w <= 0 or h <= 0:
                continue
            if score == 0 or cat == 0:
                ign.append([x, y, w, h])
                continue
            if cat == 11:
                continue
            x1, y1 = max(0, x), max(0, y)
            x2, y2 = min(W, x + w), min(H, y + h)
            if x2 <= x1 or y2 <= y1:
                continue
            cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
            bw, bh = (x2 - x1) / W, (y2 - y1) / H
            out.append(f"{cat-1} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        (lbl_dir / ann.name).write_text("\n".join(out))
        ignores[ann.stem] = ign
    (split_dir / "ignore_regions.json").write_text(json.dumps(ignores))

for split, name in SPLITS.items():
    if download_split(name):
        convert(DATA / name)

TRAIN   = DATA / "VisDrone2019-DET-train"
MASKED  = DATA / "train_masked"
TILED   = DATA / "train_tiles"
TILE_SIZES = [640, 960]
OVERLAP    = 0.25
MIN_VIS    = 0.6     # keep a cut box only if >=60% of it is inside the tile
EMPTY_KEEP = 0.1     # keep 10% of tiles that contain no objects
GREY = 114
for d in (MASKED, TILED):
    (d / "images").mkdir(parents=True, exist_ok=True)
    (d / "labels").mkdir(parents=True, exist_ok=True)

IGN = json.loads((TRAIN / "ignore_regions.json").read_text())

def load_boxes(stem, W, H):
    p = TRAIN / "labels" / f"{stem}.txt"
    rows = [r.split() for r in p.read_text().splitlines() if r.strip()] if p.exists() else []
    out = []
    for c, cx, cy, w, h in rows:
        cx, cy, w, h = float(cx)*W, float(cy)*H, float(w)*W, float(h)*H
        out.append([int(c), cx-w/2, cy-h/2, cx+w/2, cy+h/2])
    return np.array(out, dtype=np.float32).reshape(-1, 5)

def starts(L, T, step):
    if L <= T:
        return [0]
    s = list(range(0, L - T + 1, step))
    if s[-1] + T < L:
        s.append(L - T)
    return s

def process(stem):
    rng = random.Random(stem)
    img = cv2.imread(str(TRAIN / "images" / f"{stem}.jpg"))
    H, W = img.shape[:2]
    for x, y, w, h in IGN.get(stem, []):
        img[max(0, y):y+h, max(0, x):x+w] = GREY
    cv2.imwrite(str(MASKED / "images" / f"{stem}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    shutil.copy(TRAIN / "labels" / f"{stem}.txt", MASKED / "labels" / f"{stem}.txt")

    boxes = load_boxes(stem, W, H)
    n = 0
    for T in TILE_SIZES:
        if T >= max(W, H):
            continue
        step = int(T * (1 - OVERLAP))
        for y0 in starts(H, T, step):
            for x0 in starts(W, T, step):
                x1, y1, x2, y2 = x0, y0, min(x0+T, W), min(y0+T, H)
                tile = img[y1:y2, x1:x2].copy()
                tw, th = x2 - x1, y2 - y1
                lines = []
                if len(boxes):
                    bx1 = np.clip(boxes[:, 1], x1, x2); by1 = np.clip(boxes[:, 2], y1, y2)
                    bx2 = np.clip(boxes[:, 3], x1, x2); by2 = np.clip(boxes[:, 4], y1, y2)
                    inter = (bx2 - bx1) * (by2 - by1)
                    area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 4] - boxes[:, 2])
                    vis = np.where(area > 0, inter / area, 0)
                    for i in np.where(inter > 0)[0]:
                        if vis[i] >= MIN_VIS:
                            cx = ((bx1[i] + bx2[i]) / 2 - x1) / tw
                            cy = ((by1[i] + by2[i]) / 2 - y1) / th
                            w_ = (bx2[i] - bx1[i]) / tw
                            h_ = (by2[i] - by1[i]) / th
                            lines.append(f"{int(boxes[i,0])} {cx:.6f} {cy:.6f} {w_:.6f} {h_:.6f}")
                        else:
                            tile[int(by1[i]-y1):int(by2[i]-y1), int(bx1[i]-x1):int(bx2[i]-x1)] = GREY
                if not lines and rng.random() > EMPTY_KEEP:
                    continue
                name = f"{stem}_t{T}_{x0}_{y0}"
                cv2.imwrite(str(TILED / "images" / f"{name}.jpg"), tile, [cv2.IMWRITE_JPEG_QUALITY, 95])
                (TILED / "labels" / f"{name}.txt").write_text("\n".join(lines))
                n += 1
    return n

stems = [p.stem for p in (TRAIN / "images").glob("*.jpg")]
with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    counts = list(tqdm(ex.map(process, stems, chunksize=16), total=len(stems), desc="masking + tiling"))
print(f"D1: {len(stems)} masked images | D2 tiles: {sum(counts)}   (K1 had 6471 | 63745)")

VAL = "VisDrone2019-DET-val/images"
TEST = "VisDrone2019-DET-test-dev/images"
names_block = "names:\n" + "\n".join(f"  {i}: {n}" for i, n in enumerate(NAMES))
(DATA / "d2_tiles.yaml").write_text(
    f"path: {DATA}\ntrain:\n  - train_masked/images\n  - train_tiles/images\nval: {VAL}\ntest: {TEST}\n{names_block}")
print("written", DATA / "d2_tiles.yaml")

VisDrone2019-DET-train: 100%|██████████| 6471/6471 [00:03<00:00, 1893.83it/s]


VisDrone2019-DET-val: 100%|██████████| 548/548 [00:00<00:00, 1808.73it/s]


masking + tiling: 100%|██████████| 6471/6471 [03:13<00:00, 33.51it/s]

D1: 6471 masked images | D2 tiles: 63745   (K1 had 6471 | 63745)
written /tmp/VisDrone/d2_tiles.yaml


In [5]:
# --- Config ---------------------------------------------------------------
from pathlib import Path

WORK = Path("/kaggle/working")
RUNS = WORK / "runs_yoloe"

# Raw VisDrone splits for scoring (from the download cell)
VAL_IMG_DIR  = "/tmp/VisDrone_raw/VisDrone2019-DET-val/images"
VAL_ANN_DIR  = "/tmp/VisDrone_raw/VisDrone2019-DET-val/annotations"
TEST_IMG_DIR = "/tmp/VisDrone_raw/VisDrone2019-DET-test-dev/images"
TEST_ANN_DIR = "/tmp/VisDrone_raw/VisDrone2019-DET-test-dev/annotations"

# K1 checkpoint: scored through the same quick scorer, and its training settings are copied for E2
K1_WEIGHTS = "/kaggle/input/notebooks/ciiphersiren/visdrone-training/runs/K1_y26m_tiles/weights/best.pt"

# Training tiles: built by the "Build K1's training data" cell (same data K1 trained on)
DATA_YAML = "/tmp/VisDrone/d2_tiles.yaml"

MODEL_YAML = "yoloe-26m.yaml"        # detection head config
PRETRAINED = "yoloe-26m-seg.pt"      # official checkpoint (auto-downloads; needs Internet ON)

DEVICE = "0,1"     # DDP on 2×T4. If DDP misbehaves in the notebook, use "0".
IMGSZ_TRAIN = 1024
EVAL_SIZES = [1536]          # full-image eval size(s); K1 headline is full@1536
SAHI_SLICE = 512

# K1's batch / augmentation, read from its checkpoint so E2 trains under the same recipe
K1_HP = k1_hp_from_ckpt(K1_WEIGHTS)
print("K1_HP =", K1_HP)
E1_EPOCHS, E1_HOURS = 5, 3.0
E2_EPOCHS, E2_HOURS = 12, 10.5     # same budget as K1

VD_CLASSES = ["pedestrian", "people", "bicycle", "car", "van",
              "truck", "tricycle", "awning-tricycle", "bus", "motor"]
RESULTS = []   # every scored run appends a row here

K1_HP = {'batch': 16, 'workers': 4, 'mosaic': 1.0, 'close_mosaic': 3, 'mixup': 0.1, 'copy_paste': 0.0, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'erasing': 0.4, 'multi_scale': 0.0, 'cos_lr': True, 'lrf': 0.01}


In [6]:
# --- Helpers: GT loading, quick scorer, full-image + SAHI prediction, latency ---
import contextlib, io, shutil, time, json
import numpy as np, torch, torchvision
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO, YOLOE

CUDA = torch.cuda.is_available()
DEV0 = 0 if CUDA else "cpu"
HALF = CUDA

def load_visdrone_gt(img_dir, ann_dir):
    # VisDrone txt: x,y,w,h,score,category,truncation,occlusion
    # cat 0 = ignored region, 11 = others (not evaluated), 1..10 = classes
    images, anns, ignores, aid = [], [], {}, 1
    for i, p in enumerate(sorted(Path(img_dir).glob("*.jpg"))):
        w, h = Image.open(p).size
        images.append(dict(id=i, file_name=p.name, width=w, height=h))
        ig, objs = [], []
        ann = Path(ann_dir) / f"{p.stem}.txt"
        if ann.exists():
            for line in ann.read_text().splitlines():
                v = [int(float(x)) for x in line.strip().rstrip(",").split(",")[:6]] if line.strip() else None
                if not v: continue
                x, y, bw, bh, _, cat = v
                if bw <= 0 or bh <= 0: continue
                if cat == 0: ig.append([x, y, x + bw, y + bh])
                elif 1 <= cat <= 10: objs.append([x, y, bw, bh, cat])
        ig = np.array(ig, dtype=np.float32).reshape(-1, 4)
        ignores[i] = ig
        for x, y, bw, bh, cat in objs:
            if len(ig) and _ign_frac(np.array([[x, y, x + bw, y + bh]], np.float32), ig)[0] > 0.5:
                continue   # GT inside ignore region is dropped, as in the official toolkit
            anns.append(dict(id=aid, image_id=i, category_id=cat, bbox=[x, y, bw, bh], area=bw * bh, iscrowd=0)); aid += 1
    gt = dict(images=images, annotations=anns,
              categories=[dict(id=k + 1, name=n) for k, n in enumerate(VD_CLASSES)])
    return gt, ignores

def _ign_frac(boxes, ig):
    # fraction of each box's area covered by any single ignore region (max over regions)
    if len(ig) == 0 or len(boxes) == 0: return np.zeros(len(boxes))
    x1 = np.maximum(boxes[:, None, 0], ig[None, :, 0]); y1 = np.maximum(boxes[:, None, 1], ig[None, :, 1])
    x2 = np.minimum(boxes[:, None, 2], ig[None, :, 2]); y2 = np.minimum(boxes[:, None, 3], ig[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = np.clip((boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]), 1e-6, None)
    return (inter / area[:, None]).max(1)

def quick_score(gt, ignores, dets, label=None, **meta):
    # dets: list of dict(image_id, category_id 1..10, bbox xywh, score). Drops dets >50% inside ignore regions.
    keep = []
    by_img = {}
    for d in dets: by_img.setdefault(d["image_id"], []).append(d)
    for iid, ds in by_img.items():
        b = np.array([[d["bbox"][0], d["bbox"][1], d["bbox"][0] + d["bbox"][2], d["bbox"][1] + d["bbox"][3]] for d in ds], np.float32)
        frac = _ign_frac(b, ignores.get(iid, np.zeros((0, 4))))
        keep += [d for d, f in zip(ds, frac) if f <= 0.5]
    out = dict(run=label, **meta, AP=0.0, AP50=0.0, AP75=0.0, APs=0.0, APm=0.0, APl=0.0)
    if keep:
        with contextlib.redirect_stdout(io.StringIO()):
            G = COCO(); G.dataset = gt; G.createIndex()
            E = COCOeval(G, G.loadRes(keep), "bbox")
            E.params.maxDets = [1, 100, 500]   # VisDrone evaluates up to 500 dets/image
            E.evaluate(); E.accumulate(); E.summarize()
        s = E.stats
        out.update(AP=s[0], AP50=s[1], AP75=s[2], APs=s[3], APm=s[4], APl=s[5])
        p = E.eval["precision"][0, :, :, 0, 2]   # IoU .5, all area, maxDets 500
        for k, n in enumerate(VD_CLASSES):
            pk = p[:, k]; out[f"AP50_{n}"] = float(pk[pk > -1].mean()) if (pk > -1).any() else 0.0
    if label: RESULTS.append(out)
    print(f"{label}: AP50={out['AP50']:.3f}  AP={out['AP']:.3f}  APs={out['APs']:.3f}  ({len(keep)} dets kept)")
    return out

def _merge_same_class(dets, iou=0.6):
    # used when several prompts map to one class: suppress duplicates across synonyms
    if not dets: return dets
    b = torch.tensor([[d["bbox"][0], d["bbox"][1], d["bbox"][0] + d["bbox"][2], d["bbox"][1] + d["bbox"][3]] for d in dets])
    s = torch.tensor([d["score"] for d in dets]); c = torch.tensor([d["category_id"] for d in dets])
    k = torchvision.ops.batched_nms(b.float(), s.float(), c, iou)
    return [dets[i] for i in k.tolist()]

def predict_full(model, gt, img_dir, imgsz, cls_map=None, conf=0.001, max_det=1000, save_txt_dir=None):
    # cls_map: model class index -> VisDrone category id (1..10). None = identity (+1).
    dets, merge = [], cls_map is not None and len(set(cls_map.values())) < len(cls_map)
    if save_txt_dir: Path(save_txt_dir).mkdir(parents=True, exist_ok=True)
    for im in gt["images"]:
        r = model.predict(str(Path(img_dir) / im["file_name"]), imgsz=imgsz, conf=conf, max_det=max_det,
                          half=HALF, device=DEV0, verbose=False)[0]
        xyxy = r.boxes.xyxy.cpu().numpy(); sc = r.boxes.conf.cpu().numpy(); cl = r.boxes.cls.cpu().numpy().astype(int)
        ds = [dict(image_id=im["id"], category_id=(cls_map[c] if cls_map else c + 1),
                   bbox=[float(a), float(b), float(c2 - a), float(d - b)], score=float(s))
              for (a, b, c2, d), s, c in zip(xyxy, sc, cl)]
        if merge: ds = _merge_same_class(ds)
        dets += ds
        if save_txt_dir:   # VisDrone result format, for the eval notebook
            with open(Path(save_txt_dir) / f"{Path(im['file_name']).stem}.txt", "w") as f:
                for d in ds:
                    x, y, w, h = d["bbox"]; f.write(f"{x:.1f},{y:.1f},{w:.1f},{h:.1f},{d['score']:.4f},{d['category_id']},-1,-1\n")
    return dets

def predict_sahi(model, gt, img_dir, cls_map=None, slice_size=SAHI_SLICE, overlap=0.2, conf=0.001):
    # Same defaults as SAHI's get_sliced_prediction (full-image pass on, GREEDYNMM / IOS 0.5).
    # Match these to your eval notebook's SAHI settings if they differ.
    from sahi import AutoDetectionModel
    from sahi.predict import get_sliced_prediction
    sm = AutoDetectionModel.from_pretrained(model_type="ultralytics", model=model, confidence_threshold=conf,
                                            device="cuda:0" if CUDA else "cpu", image_size=slice_size)
    dets, merge = [], cls_map is not None and len(set(cls_map.values())) < len(cls_map)
    for im in gt["images"]:
        res = get_sliced_prediction(str(Path(img_dir) / im["file_name"]), sm, slice_height=slice_size, slice_width=slice_size,
                                    overlap_height_ratio=overlap, overlap_width_ratio=overlap, verbose=0)
        ds = []
        for op in res.object_prediction_list:
            c = op.category.id; b = op.bbox
            ds.append(dict(image_id=im["id"], category_id=(cls_map[c] if cls_map else c + 1),
                           bbox=[b.minx, b.miny, b.maxx - b.minx, b.maxy - b.miny], score=float(op.score.value)))
        if merge: ds = _merge_same_class(ds)
        dets += ds
    return dets

def latency_ms(model, img_dir, imgsz, n=100):
    paths = sorted(Path(img_dir).glob("*.jpg"))[:n]
    for p in paths[:5]: model.predict(str(p), imgsz=imgsz, half=HALF, device=DEV0, verbose=False)   # warm-up
    if CUDA: torch.cuda.synchronize()
    t = time.perf_counter()
    for p in paths: model.predict(str(p), imgsz=imgsz, half=HALF, device=DEV0, verbose=False)
    if CUDA: torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - t) / len(paths)

def promptable(weights, prompts):
    m = YOLOE(weights); m.set_classes(list(prompts)); return m

VAL_GT, VAL_IGN = load_visdrone_gt(VAL_IMG_DIR, VAL_ANN_DIR)
print(len(VAL_GT["images"]), "val images,", len(VAL_GT["annotations"]), "GT boxes")

548 val images, 38610 GT boxes


In [7]:
# --- K1 through the same quick scorer (calibration row) -------------------
# If this lands near your eval-notebook numbers (AP50 0.625 / AP 0.399 at full@1536), the quick scorer is trustworthy.
if K1_WEIGHTS and Path(K1_WEIGHTS).exists():
    k1 = YOLO(K1_WEIGHTS)
    for sz in EVAL_SIZES:
        quick_score(VAL_GT, VAL_IGN, predict_full(k1, VAL_GT, VAL_IMG_DIR, sz), label="K1 YOLO26m", mode=f"full@{sz}",
                    ms=latency_ms(k1, VAL_IMG_DIR, sz))
    del k1; torch.cuda.empty_cache()

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

In [8]:
PROMPT_SETS = {
    "names": {c: [c] for c in VD_CLASSES},
    "descriptive": {
        "pedestrian": ["person walking or standing"], "people": ["person sitting"],
        "bicycle": ["bicycle"], "car": ["car"], "van": ["van"], "truck": ["truck"],
        "tricycle": ["tricycle"], "awning-tricycle": ["tricycle with a canopy"],
        "bus": ["bus"], "motor": ["motorcycle"]},
    "synonyms": {
        "pedestrian": ["pedestrian", "person walking"], "people": ["person sitting", "seated person"],
        "bicycle": ["bicycle"], "car": ["car"], "van": ["van", "minivan"], "truck": ["truck", "lorry"],
        "tricycle": ["tricycle", "cargo tricycle"], "awning-tricycle": ["tricycle with a canopy", "auto rickshaw"],
        "bus": ["bus"], "motor": ["motorcycle", "scooter"]},
    "aerial": {c: [f"aerial view of a {p}"] for c, p in zip(VD_CLASSES,
        ["pedestrian", "person sitting", "bicycle", "car", "van", "truck", "tricycle", "tricycle with a canopy", "bus", "motorcycle"])},
}

def flatten(ps):
    prompts, cls_map = [], {}
    for k, c in enumerate(VD_CLASSES):
        for p in ps[c]:
            cls_map[len(prompts)] = k + 1; prompts.append(p)
    return prompts, cls_map

e0 = {}
for name, ps in PROMPT_SETS.items():
    prompts, cmap = flatten(ps)
    m = promptable(PRETRAINED, prompts)
    e0[name] = quick_score(VAL_GT, VAL_IGN, predict_full(m, VAL_GT, VAL_IMG_DIR, EVAL_SIZES[0], cls_map=cmap),
                           label=f"E0 zero-shot [{name}]", mode=f"full@{EVAL_SIZES[0]}")
    del m; torch.cuda.empty_cache()

BEST_PS = max(e0, key=lambda k: e0[k]["AP50"])
print("best prompt set:", BEST_PS)

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

In [9]:
# E0 with the best prompt set: SAHI-512, latency, and save a promptable checkpoint for later stages.
prompts, cmap = flatten(PROMPT_SETS[BEST_PS])
m = promptable(PRETRAINED, prompts)
e0[BEST_PS]["ms"] = latency_ms(m, VAL_IMG_DIR, EVAL_SIZES[0])   # updates the full-image row in RESULTS
quick_score(VAL_GT, VAL_IGN, predict_sahi(m, VAL_GT, VAL_IMG_DIR, cls_map=cmap),
            label=f"E0 zero-shot [{BEST_PS}]", mode=f"SAHI-{SAHI_SLICE}")
del m; torch.cuda.empty_cache()
# The pretrained file itself stays the open-vocab model: load it and call set_classes() with any prompts.

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

2026-10-12 05:58:51,945 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 05:58:54,011 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 05:58:55,345 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 05:58:56,515 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 05:58:58,488 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 05:59:00,162 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 05:59:01,876 - sahi - WARNING - Switching postprocess type/metric

E0 zero-shot [descriptive]: AP50=0.226  AP=0.115  APs=0.048  (245763 dets kept)


In [10]:
import pandas as pd
pd.set_option("display.float_format", "{:.3f}".format)
pd.DataFrame(RESULTS).drop(columns=["APm", "APl"], errors="ignore").T

,0,1,2,3,4,5
run,K1 YOLO26m,E0 zero-shot [names],E0 zero-shot [descriptive],E0 zero-shot [synonyms],E0 zero-shot [aerial],E0 zero-shot [descriptive]
mode,full@1536,full@1536,full@1536,full@1536,full@1536,SAHI-512
ms,37.470,NaN,57.142,NaN,NaN,NaN
AP,0.401,0.128,0.153,0.146,0.078,0.115
AP50,0.628,0.200,0.261,0.249,0.127,0.226
AP75,0.425,0.137,0.152,0.146,0.081,0.105
APs,0.317,0.065,0.084,0.081,0.038,0.048
AP50_pedestrian,0.762,0.015,0.352,0.219,0.107,0.287
AP50_people,0.630,0.147,0.122,0.132,0.123,0.114
AP50_bicycle,0.488,0.142,0.146,0.147,0.094,0.112


In [11]:
import os
# SAHI (used in the E0 / eval cells) sets CUDA_VISIBLE_DEVICES="0" for the whole session, which hides
# the second T4 from the DDP worker processes. Undo it before multi-GPU training.
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
assert DATA_YAML and Path(DATA_YAML).exists(), "Run the Build K1's training data cell first"
from ultralytics.models.yolo.yoloe import YOLOEPETrainer

def lin_probe_freeze(yaml):
    m = YOLOE(yaml); hi = len(m.model.model) - 1; head = m.model.model[-1]
    fr = [str(i) for i in range(hi)]
    for n, _ in head.named_children():
        if n not in ("cv3", "one2one_cv3"): fr.append(f"{hi}.{n}")
    for br in ("cv3", "one2one_cv3"):
        if getattr(head, br, None) is not None:
            for s in range(len(getattr(head, br))):
                fr += [f"{hi}.{br}.{s}.0", f"{hi}.{br}.{s}.1"]
    return fr

def report_trainable(trainer):
    n_t = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    n = sum(p.numel() for p in trainer.model.parameters())
    print(f"trainable params: {n_t:,} / {n:,}")

m = YOLOE(MODEL_YAML); m.load(PRETRAINED)
m.add_callback("on_train_start", report_trainable)   # should be a tiny fraction for E1
m.train(data=DATA_YAML, trainer=YOLOEPETrainer, freeze=lin_probe_freeze(MODEL_YAML),
        epochs=E1_EPOCHS, time=E1_HOURS, imgsz=IMGSZ_TRAIN, device=DEVICE,
        optimizer="AdamW", lr0=1e-3, warmup_bias_lr=0.0, weight_decay=0.025, momentum=0.9,
        project=str(RUNS), name="e1_linprobe", exist_ok=True, save_period=1,
        **{**K1_HP, "close_mosaic": 0})
E1_W = WORK / "yoloe26m_e1_linprobe.pt"   # keep 'yoloe' in the filename so YOLO() auto-loads it as YOLOE
shutil.copy(RUNS / "e1_linprobe/weights/best.pt", E1_W)

Transferred 882/882 items from pretrained weights
New https://pypi.org/project/ultralytics/8.4.175 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/VisDrone/d2_tiles.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '1

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
        2/7      4.89G      1.861       1.75   0.003507         51       1024: 100% ━━━━━━━━━━━━ 4389/4389 2.8it/s 26:00
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.0it/s 8.9s
                   all        548      38759      0.441      0.361      0.346      0.192

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
        3/7      6.01G      1.859      1.734   0.003513         52       1024: 100% ━━━━━━━━━━━━ 4389/4389 2.9it/s 25:31
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.1it/s 8.6s
                   all        548      38759      0.445      0.363      0.349      0.194

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
        4/7      7.16G      1.862      1.729     0.0035        103       1024: 100% ━━━━━

PosixPath('/kaggle/working/yoloe26m_e1_linprobe.pt')

In [12]:
# E1 eval (closed-set: classes are baked in, in DATA_YAML order)
m = YOLOE(E1_W)
assert list(m.names.values()) == VD_CLASSES, m.names
for sz in EVAL_SIZES:
    quick_score(VAL_GT, VAL_IGN, predict_full(m, VAL_GT, VAL_IMG_DIR, sz), label="E1 lin-probe", mode=f"full@{sz}",
                ms=latency_ms(m, VAL_IMG_DIR, sz))
quick_score(VAL_GT, VAL_IGN, predict_sahi(m, VAL_GT, VAL_IMG_DIR), label="E1 lin-probe", mode=f"SAHI-{SAHI_SLICE}")
del m; torch.cuda.empty_cache()

import pandas as pd
pd.DataFrame(RESULTS).to_csv(WORK / "e1_results.csv", index=False)   # RESULTS is lost when the session ends
print("saved", WORK / "e1_results.csv")

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

2026-10-12 09:12:08,823 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)


E1 lin-probe: AP50=0.338  AP=0.188  APs=0.111  (386205 dets kept)


2026-10-12 09:12:09,469 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 09:12:09,847 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 09:12:10,223 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 09:12:10,664 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 09:12:11,049 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 09:12:11,431 - sahi - WARNING - Switching postprocess type/metric to NMS/IOU since model confidence threshold is low (0.001). (predict.py:327)
2026-10-12 09:12:11,805 - sahi - WARNING - Switching postprocess type/metric

E1 lin-probe: AP50=0.319  AP=0.169  APs=0.094  (450971 dets kept)
saved /kaggle/working/e1_results.csv
